In [1]:
import re
import nltk
import numpy as np

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# Download required NLTK resources

In [2]:
nltk.download('punkt')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/satyamsharma/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/satyamsharma/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Preprocessing Module

In [5]:
class PreprocessingModule:
    """
    Handles text preprocessing tasks:
    - Lowercasing
    - Cleaning punctuation
    - Tokenization
    - Stopword removal
    """

    def __init__(self, lowercase=True, remove_stopwords=True):

        self.lowercase = lowercase
        self.remove_stopwords = remove_stopwords

        self.stop_words = set(stopwords.words('english'))

    def transform(self, text):

        # -------------------------------
        # Edge Case Handling
        # -------------------------------

        if not text or text.strip() == "":
            raise ValueError("Input text is empty.")

        if len(text.strip()) == 1:
            raise ValueError("Single-character query is not meaningful.")

        if re.fullmatch(r'[\W\d_]+', text):
            raise ValueError("Input contains only numbers or symbols.")

        # -------------------------------
        # Lowercasing
        # -------------------------------

        if self.lowercase:
            text = text.lower()

        # -------------------------------
        # Remove punctuation and numbers
        # -------------------------------

        text = re.sub(r'[^a-zA-Z\s]', '', text)

        # -------------------------------
        # Tokenization
        # -------------------------------

        tokens = word_tokenize(text)

        # -------------------------------
        # Stopword Removal
        # -------------------------------

        if self.remove_stopwords:
            tokens = [
                word for word in tokens
                if word not in self.stop_words
            ]

        return " ".join(tokens)


# Vectorizer Module

In [6]:
class VectorizerModule:
    """
    Handles:
    - TF-IDF vectorization
    - Query transformation
    - Similarity calculation
    """

    def __init__(self):

        self.vectorizer = TfidfVectorizer()
        self.document_vectors = None

    def fit(self, corpus):

        self.document_vectors = self.vectorizer.fit_transform(corpus)

    def transform(self, query):

        return self.vectorizer.transform([query])

    def similarity(self, query_vector):

        return cosine_similarity(
            query_vector,
            self.document_vectors
        )[0]


# Pipeline Module

In [7]:
class Pipeline:
    """
    Complete NLP pipeline:
    preprocessing -> vectorization -> similarity ranking
    """

    def __init__(self):

        self.preprocessor = PreprocessingModule()
        self.vectorizer = VectorizerModule()

    def run(self, query, corpus):

       
        # Preprocess Corpus
        

        processed_corpus = []

        for doc in corpus:
            cleaned_doc = self.preprocessor.transform(doc)
            processed_corpus.append(cleaned_doc)


        # Fit Vectorizer

        self.vectorizer.fit(processed_corpus)


        # Preprocess Query
    
        processed_query = self.preprocessor.transform(query)

      
        # Vectorize Query
       

        query_vector = self.vectorizer.transform(processed_query)

       
        # Compute Similarity
    

        similarity_scores = self.vectorizer.similarity(query_vector)

       
        # Rank Results
       
        ranked_results = sorted(
            enumerate(similarity_scores),
            key=lambda x: x[1],
            reverse=True
        )

        return ranked_results

# Sample Corpus (15 Documents)

In [8]:
corpus = [

    "Artificial intelligence is transforming healthcare.",

    "Machine learning models learn from data.",

    "Deep learning uses neural networks.",

    "Python is widely used in AI development.",

    "Natural language processing helps computers understand text.",

    "Transformers power modern language models.",

    "Vector databases store embeddings efficiently.",

    "Semantic search understands meaning instead of keywords.",

    "Large language models generate human-like responses.",

    "Chatbots are used in customer support.",

    "Computer vision helps machines interpret images.",

    "Data science combines statistics and programming.",

    "Recommendation systems improve user experience.",

    "AI pipelines automate machine learning workflows.",

    "Speech recognition converts audio into text."
]


In [9]:
queries = [

    "AI in healthcare",

    "language models",

    "image recognition",

    "machine learning workflow",

    "semantic understanding"
]

# Initialize Pipeline
pipeline = Pipeline()


# Run Pipeline on Queries
for query in queries:

    print("\n" + "=" * 70)
    print(f"QUERY: {query}")
    print("=" * 70)

    results = pipeline.run(query, corpus)

    # Show Top 5 Results
    for rank, (index, score) in enumerate(results[:5], start=1):

        print(f"\nRank #{rank}")
        print(f"Document Index : {index}")
        print(f"Similarity     : {score:.4f}")
        print(f"Document Text  : {corpus[index]}")



QUERY: AI in healthcare

Rank #1
Document Index : 0
Similarity     : 0.3775
Document Text  : Artificial intelligence is transforming healthcare.

Rank #2
Document Index : 3
Similarity     : 0.2681
Document Text  : Python is widely used in AI development.

Rank #3
Document Index : 13
Similarity     : 0.2519
Document Text  : AI pipelines automate machine learning workflows.

Rank #4
Document Index : 1
Similarity     : 0.0000
Document Text  : Machine learning models learn from data.

Rank #5
Document Index : 2
Similarity     : 0.0000
Document Text  : Deep learning uses neural networks.

QUERY: language models

Rank #1
Document Index : 5
Similarity     : 0.5347
Document Text  : Transformers power modern language models.

Rank #2
Document Index : 8
Similarity     : 0.4805
Document Text  : Large language models generate human-like responses.

Rank #3
Document Index : 1
Similarity     : 0.2845
Document Text  : Machine learning models learn from data.

Rank #4
Document Index : 4
Similarity   